# Module 01: NumPy for Machine Learning
## Notebook 05: Practical Machine Learning Algorithms from Scratch

Welcome to the capstone notebook for Module 01! The best way to truly understand Machine Learning is to build the algorithms from scratch using raw mathematical principles and NumPy arrays.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Build reusable, production-style feature transformers (`StandardScaler`) adhering to Scikit-Learn's `.fit()` / `.transform()` paradigm.
2. Compute **vectorized pairwise distance matrices** without loops using quadratic expansion.
3. Construct an efficient **Mini-Batch Data Generator** for stochastic optimization.
4. Implement **Linear Regression with Batch Gradient Descent** from scratch and track loss convergence.
5. Implement **Logistic Regression** with cross-entropy loss and compute evaluation metrics (accuracy, precision, recall) using boolean operations.

In [1]:
import numpy as np
import time

print(f"NumPy version: {np.__version__}")

NumPy version: 2.5.3


### 1. Feature Preprocessing: `StandardScaler` from Scratch

In machine learning, gradient-based optimizers and distance-based algorithms are sensitive to feature scales.
A proper transformer must:
1. Compute $\mu$ and $\sigma$ strictly on **training data** during `.fit()` to prevent **data leakage**.
2. Apply the learned $\mu$ and $\sigma$ to both training and test data during `.transform()`.
3. Handle zero standard deviation safely with numerical tolerance $\epsilon = 10^{-8}$.

In [2]:
class StandardScalerFromScratch:
    # Standardizes features by removing the mean and scaling to unit variance
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.scale_ = None
        
    def fit(self, X):
        # Compute mean and standard deviation along axis 0 (features)
        self.mean_ = np.mean(X, axis=0, keepdims=True)
        self.scale_ = np.std(X, axis=0, keepdims=True)
        # Prevent division by zero if a feature is constant
        self.scale_ = np.where(self.scale_ < self.eps, 1.0, self.scale_)
        return self
        
    def transform(self, X):
        if self.mean_ is None or self.scale_ is None:
            raise RuntimeError("Transformer has not been fitted yet!")
        return (X - self.mean_) / self.scale_
        
    def fit_transform(self, X):
        return self.fit(X).transform(X)
        
    def inverse_transform(self, X_scaled):
        return (X_scaled * self.scale_) + self.mean_

# Test the scaler
X_train = np.array([
    [10.0, 50000.0],
    [20.0, 70000.0],
    [30.0, 90000.0]
])

scaler = StandardScalerFromScratch()
X_scaled = scaler.fit_transform(X_train)

print("Original Features:\n", X_train)
print("\nScaled Features (zero mean, unit variance):\n", np.round(X_scaled, 4))
print("\nInverted Back to Original:\n", scaler.inverse_transform(X_scaled))

Original Features:
 [[1.e+01 5.e+04]
 [2.e+01 7.e+04]
 [3.e+01 9.e+04]]

Scaled Features (zero mean, unit variance):
 [[-1.2247 -1.2247]
 [ 0.      0.    ]
 [ 1.2247  1.2247]]

Inverted Back to Original:
 [[1.e+01 5.e+04]
 [2.e+01 7.e+04]
 [3.e+01 9.e+04]]


---
### 2. Vectorized Pairwise Distance Computation (No Loops!)

Given a test matrix $A$ of shape $(M, D)$ and training matrix $B$ of shape $(N, D)$, how do we compute the Euclidean distance between all pairs without $O(M \times N)$ Python loops?

#### The Quadratic Expansion Trick:
The squared Euclidean distance between two vectors $a$ and $b$ is:
$$\|a - b\|^2 = \|a\|^2 - 2 (a \cdot b) + \|b\|^2$$

In matrix form:
- Matrix of squared norms of rows in $A$: shape $(M, 1)$
- Matrix of squared norms of rows in $B$: shape $(1, N)$
- Cross-term $2 A B^T$: shape $(M, N)$ via `@`

Using broadcasting:
$$\text{Distances}^2 = \text{norms}_A + \text{norms}_B - 2 A B^T$$

In [3]:
def pairwise_distances_vectorized(A, B):
    # Computes Euclidean distances between M vectors in A and N vectors in B.
    # A: shape (M, D), B: shape (N, D), Returns: shape (M, N) distance matrix
    
    # Sum of squares along feature axis
    A_sq = np.sum(A ** 2, axis=1, keepdims=True)  # Shape (M, 1)
    B_sq = np.sum(B ** 2, axis=1, keepdims=True).T # Shape (1, N)
    
    # Cross product matrix
    cross_term = 2.0 * (A @ B.T)  # Shape (M, N)
    
    # Broadcasted sum
    dist_sq = A_sq + B_sq - cross_term
    
    # Clip negative values caused by floating point precision errors
    dist_sq = np.maximum(dist_sq, 0.0)
    return np.sqrt(dist_sq)

# Performance Benchmark: 200 test samples vs 1,000 training samples
rng = np.random.default_rng(42)
A_test = rng.normal(0, 1, size=(200, 10))
B_train = rng.normal(0, 1, size=(1000, 10))

start = time.time()
dist_matrix = pairwise_distances_vectorized(A_test, B_train)
dur = time.time() - start

print(f"Calculated {dist_matrix.shape[0]} x {dist_matrix.shape[1]} pairwise distances in {dur:.4f} seconds!")
print("Distance matrix shape:", dist_matrix.shape)

Calculated 200 x 1000 pairwise distances in 0.0572 seconds!
Distance matrix shape: (200, 1000)


---
### 3. Mini-Batch Data Generator

In Deep Learning and large-scale optimization, full-batch gradient descent is too slow and memory intensive. We split data into randomized mini-batches.

In [4]:
def get_batches(X, y, batch_size=32, shuffle=True, seed=None):
    # Yields mini-batches of features and labels
    num_samples = len(X)
    rng = np.random.default_rng(seed)
    
    indices = np.arange(num_samples)
    if shuffle:
        rng.shuffle(indices)
        
    for start_idx in range(0, num_samples, batch_size):
        batch_idx = indices[start_idx : start_idx + batch_size]
        yield X[batch_idx], y[batch_idx]

# Demonstrate batch generation
X_dummy = np.arange(100).reshape(50, 2)
y_dummy = np.arange(50)

print("Generating batches of size 16 from 50 samples:")
for batch_num, (xb, yb) in enumerate(get_batches(X_dummy, y_dummy, batch_size=16, shuffle=False)):
    print(f"  Batch {batch_num + 1}: X shape = {xb.shape}, y shape = {yb.shape}")

Generating batches of size 16 from 50 samples:
  Batch 1: X shape = (16, 2), y shape = (16,)
  Batch 2: X shape = (16, 2), y shape = (16,)
  Batch 3: X shape = (16, 2), y shape = (16,)
  Batch 4: X shape = (2, 2), y shape = (2,)


---
### 4. Linear Regression with Batch Gradient Descent

#### Mathematical Formulation:
- **Hypothesis**: $\hat{\mathbf{y}} = X \mathbf{w} + b$
- **Loss Function (MSE)**:
  $$J(\mathbf{w}, b) = \frac{1}{2m} \sum_{i=1}^m (\hat{y}_i - y_i)^2 = \frac{1}{2m} (\hat{\mathbf{y}} - \mathbf{y})^T (\hat{\mathbf{y}} - \mathbf{y})$$
- **Gradients**:
  $$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{m} X^T (\hat{\mathbf{y}} - \mathbf{y})$$
  $$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^m (\hat{y}_i - y_i)$$
- **Parameter Updates**:
  $$\mathbf{w} \leftarrow \mathbf{w} - \alpha \frac{\partial J}{\partial \mathbf{w}}$$
  $$b \leftarrow b - \alpha \frac{\partial J}{\partial b}$$

In [5]:
class LinearRegressionScratch:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []
        
    def fit(self, X, y):
        m, n = X.shape
        self.weights = np.zeros(n)
        self.bias = 0.0
        self.loss_history = []
        
        for i in range(self.n_iterations):
            # 1. Forward pass (predictions)
            y_pred = X @ self.weights + self.bias
            
            # 2. Compute Mean Squared Error
            loss = (1.0 / (2.0 * m)) * np.sum((y_pred - y) ** 2)
            self.loss_history.append(loss)
            
            # 3. Backward pass (analytical gradients)
            error = y_pred - y
            dw = (1.0 / m) * (X.T @ error)
            db = (1.0 / m) * np.sum(error)
            
            # 4. Parameter update step
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
        return self
        
    def predict(self, X):
        return X @ self.weights + self.bias

# Synthetic Dataset
rng = np.random.default_rng(42)
X_syn = rng.normal(0, 1, size=(200, 3))
true_w = np.array([2.5, -1.8, 0.7])
true_b = 4.2
y_syn = X_syn @ true_w + true_b + rng.normal(0, 0.2, size=200)

# Train the model
model = LinearRegressionScratch(learning_rate=0.05, n_iterations=300)
model.fit(X_syn, y_syn)

print("True weights:     ", true_w, "| True bias:", true_b)
print("Learned weights:  ", np.round(model.weights, 4), "| Learned bias:", round(model.bias, 4))
print(f"Initial Loss: {model.loss_history[0]:.4f} -> Final Loss: {model.loss_history[-1]:.4f}")

# Compute R2 score
y_pred = model.predict(X_syn)
ss_res = np.sum((y_syn - y_pred) ** 2)
ss_tot = np.sum((y_syn - np.mean(y_syn)) ** 2)
r2_score = 1 - (ss_res / ss_tot)
print(f"Model R-squared (R2) Score: {r2_score:.4f}")

True weights:      [ 2.5 -1.8  0.7] | True bias: 4.2
Learned weights:   [ 2.5074 -1.8015  0.6897] | Learned bias: 4.1926
Initial Loss: 13.8250 -> Final Loss: 0.0205
Model R-squared (R2) Score: 0.9954


---
### 5. Binary Logistic Regression from Scratch

#### Mathematical Formulation:
- **Linear Combination**: $z = X \mathbf{w} + b$
- **Sigmoid Activation**: $\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$
- **Binary Cross-Entropy Loss**:
  $$J(\mathbf{w}, b) = -\frac{1}{m} \sum_{i=1}^m \left[ y_i \ln(\hat{y}_i) + (1 - y_i) \ln(1 - \hat{y}_i) \right]$$
- **Gradients** (miraculously identical form to linear regression, but with sigmoid predictions!):
  $$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{m} X^T (\hat{\mathbf{y}} - \mathbf{y})$$
  $$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^m (\hat{y}_i - y_i)$$

In [6]:
class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.1, n_iterations=500):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []
        
    @staticmethod
    def _sigmoid(z):
        # Clip z to prevent numeric overflow in exp(-z)
        z_safe = np.clip(z, -250.0, 250.0)
        return 1.0 / (1.0 + np.exp(-z_safe))
        
    def fit(self, X, y):
        m, n = X.shape
        self.weights = np.zeros(n)
        self.bias = 0.0
        self.loss_history = []
        
        for _ in range(self.n_iterations):
            # Forward pass
            z = X @ self.weights + self.bias
            y_pred = self._sigmoid(z)
            
            # Loss computation (with epsilon clipping for log stability)
            eps = 1e-15
            y_pred_clipped = np.clip(y_pred, eps, 1.0 - eps)
            loss = - (1.0 / m) * np.sum(y * np.log(y_pred_clipped) + (1.0 - y) * np.log(1.0 - y_pred_clipped))
            self.loss_history.append(loss)
            
            # Backward pass
            error = y_pred - y
            dw = (1.0 / m) * (X.T @ error)
            db = (1.0 / m) * np.sum(error)
            
            # Update parameters
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
        return self
        
    def predict_proba(self, X):
        z = X @ self.weights + self.bias
        return self._sigmoid(z)
        
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(np.int32)

# Generate synthetic binary classification dataset
rng = np.random.default_rng(101)
X_pos = rng.normal(loc=1.5, scale=0.8, size=(100, 2))
X_neg = rng.normal(loc=-1.5, scale=0.8, size=(100, 2))
X_cls = np.vstack([X_pos, X_neg])
y_cls = np.array([1] * 100 + [0] * 100)

clf = LogisticRegressionScratch(learning_rate=0.2, n_iterations=400)
clf.fit(X_cls, y_cls)

y_pred_cls = clf.predict(X_cls)

# Compute classification evaluation metrics using pure NumPy boolean operations
accuracy = np.mean(y_pred_cls == y_cls)
true_positives = np.sum((y_pred_cls == 1) & (y_cls == 1))
predicted_positives = np.sum(y_pred_cls == 1)
actual_positives = np.sum(y_cls == 1)

precision = true_positives / predicted_positives
recall = true_positives / actual_positives
f1 = 2 * (precision * recall) / (precision + recall)

print(f"Classification Metrics from Scratch:")
print(f"  Accuracy:  {accuracy * 100:.2f}%")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")

Classification Metrics from Scratch:
  Accuracy:  99.50%
  Precision: 1.0000
  Recall:    0.9900
  F1 Score:  0.9950


### Module 01 Conclusion & Congratulations!
You have successfully completed **Module 01: NumPy for Machine Learning**!

Throughout these 5 notebooks, you transitioned from basic array structures to:
1. Array creation, data types, and memory optimization (`01_array_basics_and_creation.ipynb`).
2. Slicing, memory views vs copies, and multidimensional tensor reshaping (`02_indexing_slicing_and_reshaping.ipynb`).
3. Vectorization, ufuncs, broadcasting rules, and boolean masking (`03_vectorization_and_broadcasting.ipynb`).
4. Linear algebra, eigenvalues, norms, and modern random generation (`04_math_stats_and_linear_algebra.ipynb`).
5. Feature scaling, gradient descent, linear regression, logistic regression, and KNN (`05_practical_ml_applications.ipynb`).

**Up Next:** Proceed to **Module 02: Pandas** to master tabular data exploration, cleaning, aggregation, and feature engineering!